#  RAG의 기본 개념 / 문서 전처리 과정의 이해

---

## 학습 목표
- RAG(Retrieval-Augmented Generation)의 기본 개념 이해
- 문서 로딩, 청크 분할, 임베딩, 벡터 저장 과정 실습
- LangChain을 활용한 RAG 파이프라인 구축
- 검색 기반 질의응답 시스템 구현

# 환경 설정 및 준비

In [ ]:
# 필수 라이브러리 설치
# uv add langchain langchain-openai langchain-community beautifulsoup4 langchain-chroma

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

---

## 1. RAG 개념 및 아키텍처

### 1.1 RAG란?

**RAG (Retrieval-Augmented Generation)** 는 기존 LLM의 한계를 보완하기 위한 방법론:

- **문제점**: LLM은 훈련 시점의 고정된 데이터에만 의존
- **해결책**: 외부 지식 베이스를 동적으로 검색하여 응답 생성 시 활용
- **장점**: 최신 정보, 도메인 특화 지식, 사실 기반 응답 가능

### 1.2 RAG의 핵심 구성요소

```mermaid
graph LR
    A[사용자 쿼리] --> B[문서 검색<br/>Retrieval]
    B --> C[관련 문서]
    C --> D[컨텍스트 증강<br/>Augmentation]
    A --> D
    D --> E[LLM 생성<br/>Generation]
    E --> F[최종 응답]
```

**1. 검색 (Retrieval) 시스템**
- 임베딩 모델: 텍스트를 벡터로 변환
- 벡터 데이터베이스: 임베딩 벡터 저장 및 인덱싱
- 유사도 검색: 코사인 유사도, 유클리드 거리 등

**2. 증강 (Augmentation)**
- 검색된 문서 전처리 및 포맷팅
- 프롬프트 엔지니어링
- 컨텍스트 길이 관리

**3. 생성 (Generation)**
- LLM을 통한 최종 응답 생성
- 검색된 컨텍스트와 원본 질의 결합

### 1.3 RAG vs 기존 접근법 비교

| 특성 | 기존 LLM | 파인튜닝 | RAG |
|------|----------|----------|-----|
| 최신 정보 | ❌ | ❌ | ✅ |
| 구현 복잡도 | 낮음 | 높음 | 중간 |
| 계산 비용 | 낮음 | 높음 | 중간 |
| 소스 추적 | ❌ | ❌ | ✅ |
| 환각 방지 | ❌ | △ | ✅ |

---

## 2. 최신 LangChain을 활용한 RAG 구현

### Step 1: **Indexing**

1. 문서 수집 및 전처리
2. 문서 청크 분할
3. 임베딩 생성
4. 벡터 저장소 구축

`1. 문서 데이터 로드(Load Data)`

- RAG에 사용할 데이터를 불러오는 단계 (검색에 사용될 지식이나 정보)
- 외부 데이터 소스에서 정보를 수집하고, 필요한 형식으로 변환하여 시스템에 로드

In [2]:
# Data Loader - 웹페이지 데이터 가져오기
from langchain_community.document_loaders import WebBaseLoader  # type: ignore

# 위키피디아 정책과 지침
url = 'https://ko.wikipedia.org/wiki/%EC%9C%84%ED%82%A4%EB%B0%B1%EA%B3%BC:%EC%A0%95%EC%B1%85%EA%B3%BC_%EC%A7%80%EC%B9%A8'
loader = WebBaseLoader(url)

# 웹페이지 텍스트 -> Document 객체로 변환
docs = loader.load() 

# 결과 확인
print(f"Document 개수: {len(docs)}")
print(f"Document 길이: {len(docs[0].page_content)}")
print(f"Document 내용: {docs[0].page_content[5000:6000]}")

C:\Users\dwoz\AppData\Local\Temp\ipykernel_30372\4123745373.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader  # type: ignore
d:\developer\ai\edu\modu_llm7\faq_bot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


Document 개수: 1
Document 길이: 13317
Document 내용: 동체의 규범을 총체적으로 어기고 있다면 규범 준수를 위해 좀 더 빠르게 강력한 수단을 이용해야 합니다. 특히 정책 문서에 명시된 원칙을 지키지 않는 것은 대부분의 경우 다른 사용자에게 받아들여지지 않습니다 (다른 분들에게 예외 상황임을 설득할 수 있다면 가능하기는 하지만요). 이는 당신을 포함해서 편집자 개개인이 정책과 지침을 직접 집행 및 적용한다는 것을 의미합니다.
특정 사용자가 명백히 정책에 반하는 행동을 하거나 정책과 상충되는 방식으로 지침을 어기는 경우, 특히 의도적이고 지속적으로 그런 행위를 하는 경우 해당 사용자는 관리자의 제재 조치로 일시적, 혹은 영구적으로 편집이 차단될 수 있습니다. 영어판을 비롯한 타 언어판에서는 일반적인 분쟁 해결 절차로 끝낼 수 없는 사안은 중재위원회가 개입하기도 합니다.
문서 내용
정책과 지침의 문서 내용은 처음 읽는 사용자라도 원칙과 규범을 잘 이해할 수 있도록 다음 원칙을 지켜야 합니다.
명확하게 작성하세요. 소수만 알아듣거나 준법률적인 단어, 혹은 지나치게 단순한 표현은 피해야 합니다. 명확하고, 직접적이고, 모호하지 않고, 구체적으로 작성하세요. 지나치게 상투적인 표현이나 일반론은 피하세요. 지침, 도움말 문서 및 기타 정보문 문서에서도 "해야 합니다" 혹은 "하지 말아야 합니다" 같이 직접적인 표현을 굳이 꺼릴 필요는 없습니다.
가능한 간결하게, 너무 단순하지는 않게. 정책이 중언부언하면 오해를 부릅니다. 불필요한 말은 생략하세요. 직접적이고 간결한 설명이 마구잡이식 예시 나열보다 더 이해하기 쉽습니다. 각주나 관련 문서 링크를 이용하여 더 상세히 설명할 수도 있습니다.
규칙을 만든 의도를 강조하세요. 사용자들이 상식대로 행동하리라 기대하세요. 정책의 의도가 명료하다면, 추가 설명은 필요 없죠. 즉 규칙을 '어떻게' 지키는지와 더불어 '왜' 지켜야 하는지 확실하게 밝혀야 합니다.
범위는 분명히, 중복은 피하기. 되도록 앞부분에서

In [3]:
docs[0]

Document(metadata={'source': 'https://ko.wikipedia.org/wiki/%EC%9C%84%ED%82%A4%EB%B0%B1%EA%B3%BC:%EC%A0%95%EC%B1%85%EA%B3%BC_%EC%A7%80%EC%B9%A8', 'title': '위키백과:정책과 지침 - 위키백과, 우리 모두의 백과사전', 'language': 'ko'}, page_content='\n\n\n\n위키백과:정책과 지침 - 위키백과, 우리 모두의 백과사전\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n본문으로 이동\n\n\n\n\n\n\n\n주 메뉴\n\n\n\n\n\n주 메뉴\n사이드바로 이동\n숨기기\n\n\n\n\t\t둘러보기\n\t\n\n\n대문최근 바뀜요즘 화제임의 문서로\n\n\n\n\n\n\t\t사용자 모임\n\t\n\n\n사랑방사용자 모임관리 요청\n\n\n\n\n\n\t\t편집 안내\n\t\n\n\n소개도움말정책과 지침질문방\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n검색\n\n\n\n\n\n\n\n\n\n\n\n검색\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n외관\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n기부\n\n계정 만들기\n\n로그인\n\n\n\n\n\n\n\n\n개인 도구\n\n\n\n\n\n\n기부\n\n\n계정 만들기\n\n\n로그인\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n목차\n사이드바로 이동\n숨기기\n\n\n\n\n처음 위치\n\n\n\n\n\n1\n최상위 정책\n\n\n\n\n\n\n\n\n2\n\'정책과 지침\'이란?\n\n\n\n\n\n\n\n\n3\n준수\n\n\n\n\n\n\n\n\n4\n집행\n\n\n\n\n\n\n\n\n5\n문서 내용\n\n\n\n\n\n\n\n\n6\n정책과 지침은 백과사전의 일부가 아닙

In [4]:
docs[0] # Document 객체 확인

Document(metadata={'source': 'https://ko.wikipedia.org/wiki/%EC%9C%84%ED%82%A4%EB%B0%B1%EA%B3%BC:%EC%A0%95%EC%B1%85%EA%B3%BC_%EC%A7%80%EC%B9%A8', 'title': '위키백과:정책과 지침 - 위키백과, 우리 모두의 백과사전', 'language': 'ko'}, page_content='\n\n\n\n위키백과:정책과 지침 - 위키백과, 우리 모두의 백과사전\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n본문으로 이동\n\n\n\n\n\n\n\n주 메뉴\n\n\n\n\n\n주 메뉴\n사이드바로 이동\n숨기기\n\n\n\n\t\t둘러보기\n\t\n\n\n대문최근 바뀜요즘 화제임의 문서로\n\n\n\n\n\n\t\t사용자 모임\n\t\n\n\n사랑방사용자 모임관리 요청\n\n\n\n\n\n\t\t편집 안내\n\t\n\n\n소개도움말정책과 지침질문방\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n검색\n\n\n\n\n\n\n\n\n\n\n\n검색\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n외관\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n기부\n\n계정 만들기\n\n로그인\n\n\n\n\n\n\n\n\n개인 도구\n\n\n\n\n\n\n기부\n\n\n계정 만들기\n\n\n로그인\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n목차\n사이드바로 이동\n숨기기\n\n\n\n\n처음 위치\n\n\n\n\n\n1\n최상위 정책\n\n\n\n\n\n\n\n\n2\n\'정책과 지침\'이란?\n\n\n\n\n\n\n\n\n3\n준수\n\n\n\n\n\n\n\n\n4\n집행\n\n\n\n\n\n\n\n\n5\n문서 내용\n\n\n\n\n\n\n\n\n6\n정책과 지침은 백과사전의 일부가 아닙

In [5]:
docs[0].metadata # Document의 메타데이터 확인

{'source': 'https://ko.wikipedia.org/wiki/%EC%9C%84%ED%82%A4%EB%B0%B1%EA%B3%BC:%EC%A0%95%EC%B1%85%EA%B3%BC_%EC%A7%80%EC%B9%A8',
 'title': '위키백과:정책과 지침 - 위키백과, 우리 모두의 백과사전',
 'language': 'ko'}

In [6]:
docs[0].page_content # Document의 페이지 내용 확인

'\n\n\n\n위키백과:정책과 지침 - 위키백과, 우리 모두의 백과사전\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n본문으로 이동\n\n\n\n\n\n\n\n주 메뉴\n\n\n\n\n\n주 메뉴\n사이드바로 이동\n숨기기\n\n\n\n\t\t둘러보기\n\t\n\n\n대문최근 바뀜요즘 화제임의 문서로\n\n\n\n\n\n\t\t사용자 모임\n\t\n\n\n사랑방사용자 모임관리 요청\n\n\n\n\n\n\t\t편집 안내\n\t\n\n\n소개도움말정책과 지침질문방\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n검색\n\n\n\n\n\n\n\n\n\n\n\n검색\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n외관\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n기부\n\n계정 만들기\n\n로그인\n\n\n\n\n\n\n\n\n개인 도구\n\n\n\n\n\n\n기부\n\n\n계정 만들기\n\n\n로그인\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n목차\n사이드바로 이동\n숨기기\n\n\n\n\n처음 위치\n\n\n\n\n\n1\n최상위 정책\n\n\n\n\n\n\n\n\n2\n\'정책과 지침\'이란?\n\n\n\n\n\n\n\n\n3\n준수\n\n\n\n\n\n\n\n\n4\n집행\n\n\n\n\n\n\n\n\n5\n문서 내용\n\n\n\n\n\n\n\n\n6\n정책과 지침은 백과사전의 일부가 아닙니다\n\n\n\n\n\n\n\n\n7\n채택 과정\n\n\n\n\n채택 과정 하위 문단 토글하기\n\n\n\n\n\n7.1\n제안과 채택\n\n\n\n\n\n\n\n\n7.2\n내용 변경\n\n\n\n\n\n\n7.2.1\n실질적인 변경\n\n\n\n\n\n\n\n\n\n\n7.3\n격하\n\n\n\n\n\n\n\n\n\n\n8\n같이 보기\n\n\n\n\n\n\n\n\n9\n외부 링크\n\n

In [7]:
print(docs[0].page_content)





위키백과:정책과 지침 - 위키백과, 우리 모두의 백과사전



























본문으로 이동







주 메뉴





주 메뉴
사이드바로 이동
숨기기



		둘러보기
	


대문최근 바뀜요즘 화제임의 문서로





		사용자 모임
	


사랑방사용자 모임관리 요청





		편집 안내
	


소개도움말정책과 지침질문방



















검색











검색






















외관
















기부

계정 만들기

로그인








개인 도구






기부


계정 만들기


로그인





























목차
사이드바로 이동
숨기기




처음 위치





1
최상위 정책








2
'정책과 지침'이란?








3
준수








4
집행








5
문서 내용








6
정책과 지침은 백과사전의 일부가 아닙니다








7
채택 과정




채택 과정 하위 문단 토글하기





7.1
제안과 채택








7.2
내용 변경






7.2.1
실질적인 변경










7.3
격하










8
같이 보기








9
외부 링크


















목차 토글







위키백과:정책과 지침



115개 언어




Afrikaansአማርኛअंगिकाالعربيةالدارجةمصرىঅসমীয়াАварAzərbaycancaتۆرکجهБашҡортсаBoarischБеларуская (тарашкевіца)БеларускаяБългарскиभोजपुरीBanjarবাংলাBrezhonegBosanskiCatalàНохчийнکوردیČeštinaCymraegDanskDeutschΕλληνικάEnglishEsperantoEspañolEuskaraفارسیSuomiFrançaisNordfriiskGaeilge贛語GalegoگیلکیBahasa H

`2. 문서 청크 분할(Split Texts)`

- 불러온 데이터를 작은 크기의 단위(chunk)로 분할하는 과정
- 자연어 처리(NLP) 기술을 활용하여 큰 문서를 처리가 쉽도록 문단, 문장 또는 구 단위로 나누는 작업
- 검색 효율성을 높이기 위한 중요한 과정

    1. 청크 크기 선택
        - 너무 작은 청크: 문맥 손실
        - 너무 큰 청크: 관련성 저하

    2. 중복 영역 설정
        - 문맥 유지를 위해 필요
        - 일반적으로 10-20% 권장

- LangChain 기본 Text Splitter 종류

    - **`RecursiveCharacterTextSplitter`**
        - 여러 구분자를 우선순위대로 시도하여 문서를 자연스럽게 분할
        - 기본 구분자 순서: `["\n\n", "\n", " ", ""]`
        - 대부분의 경우에 가장 좋은 성능을 보임

    - **`CharacterTextSplitter`**
        - 단일 구분자로만 분할 (예: `\n\n`)
        - 간단한 경우에 사용


- 설치: pip install langchain_text_splitters 또는 uv add langchain_text_splitters

In [8]:
# Text Split (Documents -> small chunks: Documents)
from langchain_text_splitters import CharacterTextSplitter  # type: ignore

# 1000자씩 잘라서 200자씩 겹치는 Document로 변환
text_splitter = CharacterTextSplitter(
    separator="\n\n",    # 문단 구분자
    chunk_size=1000,     # 문단 길이
    chunk_overlap=200,   # 겹치는 길이
    length_function=len, # 길이 측정 함수
    is_separator_regex=False,   # separator가 정규식인지 여부
)

splitted_docs = text_splitter.split_documents(docs)

# 결과 확인
print(f"Document 개수: {len(splitted_docs)}")
print("\n\n")

for i, doc in enumerate(splitted_docs):
    print(f"Document {i} 길이: {len(doc.page_content)}")
    print(f"Document {i} 내용: {doc.page_content[:100]}...")
    print("-"*50)

Created a chunk of size 9814, which is longer than the specified 1000


Document 개수: 5



Document 0 길이: 442
Document 0 내용: 위키백과:정책과 지침 - 위키백과, 우리 모두의 백과사전

본문으로 이동

주 메뉴

주 메뉴
사이드바로 이동
숨기기

		둘러보기
	


대문최근 바뀜요즘 화제임의 문서로

		...
--------------------------------------------------
Document 1 길이: 992
Document 1 내용: 7.3
격하


8
같이 보기


9
외부 링크


목차 토글

위키백과:정책과 지침

115개 언어


Afrikaansአማርኛअंगिकाالعربيةالدارجةمصرىঅসমী...
--------------------------------------------------
Document 2 길이: 728
Document 2 내용: 링크 편집

프로젝트 문서토론

한국어

읽기원본 보기역사 보기

도구

도구
사이드바로 이동
숨기기

		작업
	


읽기

원본 보기

역사 보기


		일반
	


여기를 가...
--------------------------------------------------
Document 3 길이: 9814
Document 3 내용: 목록
정책 목록
지침 목록
vte
위키백과의 정책(policy)과 지침(guideline)은 위키미디어 공동체가 모범적인 활동 방식, 분쟁 해결, 문서의 표준 작성 기준 등을 제시...
--------------------------------------------------
Document 4 길이: 968
Document 4 내용: 모든 정책과 지침 목록
P: 정책 목록
G: 지침 목록
가치와 원칙 요약

vte위키백과 사용자 계정비등록 사용자(IP 편집자)
계정을 만드는 것이 좋은 이유
계정 만들기
IP 사...
--------------------------------------------------


> **참고**: 위 출력에서 `Created a chunk of size 9814, which is longer than the specified 1000` 경고가 나타날 수 있습니다. 

- 이는 `chunk_size`가 엄격한 최대값이 아닌 목표값이기 때문입니다. 
- `separator="\n\n"`로 문단 단위로 분할하기 때문에, 하나의 문단이 chunk_size보다 크면 그대로 유지됩니다. 
- 이는 문맥을 유지하기 위한 의도된 동작입니다.

In [10]:
# 글자 수 기준으로 엄격하게 분할하기 
from langchain_text_splitters import CharacterTextSplitter  # type: ignore

# 1000자씩 잘라서 Document로 변환
text_splitter = CharacterTextSplitter(
    separator="",        # 문단 구분자
    chunk_size=1000,     # 문단 길이
    length_function=len, # 길이 측정 함수
    is_separator_regex=False,   # separator가 정규식인지 여부
)

equally_splitted_docs = text_splitter.split_documents(docs)

# 결과 확인
print(f"Document 개수: {len(equally_splitted_docs)}")
print("\n\n")

for i, doc in enumerate(equally_splitted_docs):
    print(f"Document {i} 길이: {len(doc.page_content)}")
    print("-"*50)

Document 개수: 17



Document 0 길이: 996
--------------------------------------------------
Document 1 길이: 1000
--------------------------------------------------
Document 2 길이: 999
--------------------------------------------------
Document 3 길이: 1000
--------------------------------------------------
Document 4 길이: 1000
--------------------------------------------------
Document 5 길이: 1000
--------------------------------------------------
Document 6 길이: 999
--------------------------------------------------
Document 7 길이: 998
--------------------------------------------------
Document 8 길이: 1000
--------------------------------------------------
Document 9 길이: 999
--------------------------------------------------
Document 10 길이: 1000
--------------------------------------------------
Document 11 길이: 1000
--------------------------------------------------
Document 12 길이: 999
--------------------------------------------------
Document 13 길이: 1000
----------------------------------------

`3. 문서 임베딩 생성(Document Embeddings)`

- 임베딩 모델을 사용하여 텍스트를 벡터로 변환
- 임베딩을 기반으로 유사성 검색에 사용
- 임베딩 모델 선택
   - 성능과 비용 고려
   - 다국어 지원 여부 확인

In [11]:
# OpenAI Embeddings - 문장 임베딩

from langchain_openai import OpenAIEmbeddings  # type: ignore

# embedding model 생성
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",  # 사용할 모델 이름을 지정 가능 
)

sample_text = "위키피디아 정책 변경 절차를 알려주세요"
embedding_vector = embedding_model.embed_query(sample_text)
print(f"임베딩 벡터의 차원: {len(embedding_vector)}")
print(f"임베딩 벡터: {embedding_vector[:10]}...")

임베딩 벡터의 차원: 1536
임베딩 벡터: [-0.00888824462890625, 0.04730224609375, 0.0105438232421875, 0.00147247314453125, -0.03497314453125, 0.04229736328125, -0.0011081695556640625, 0.0219573974609375, 0.032928466796875, -0.022308349609375]...


`4. 벡터 저장소 구축 (Vectorstores)`

- 임베딩 벡터를 벡터저장소에 저장 
- 저장된 임베딩을 기반으로 유사성 검색을 수행하는데 활용 

In [12]:
splitted_docs[0].page_content

"위키백과:정책과 지침 - 위키백과, 우리 모두의 백과사전\n\n본문으로 이동\n\n주 메뉴\n\n주 메뉴\n사이드바로 이동\n숨기기\n\n\t\t둘러보기\n\t\n\n\n대문최근 바뀜요즘 화제임의 문서로\n\n\t\t사용자 모임\n\t\n\n\n사랑방사용자 모임관리 요청\n\n\t\t편집 안내\n\t\n\n\n소개도움말정책과 지침질문방\n\n검색\n\n검색\n\n\n외관\n\n\n기부\n\n계정 만들기\n\n로그인\n\n\n개인 도구\n\n\n기부\n\n\n계정 만들기\n\n\n로그인\n\n목차\n사이드바로 이동\n숨기기\n\n\n처음 위치\n\n1\n최상위 정책\n\n\n2\n'정책과 지침'이란?\n\n\n3\n준수\n\n\n4\n집행\n\n\n5\n문서 내용\n\n\n6\n정책과 지침은 백과사전의 일부가 아닙니다\n\n\n7\n채택 과정\n\n\n채택 과정 하위 문단 토글하기\n\n7.1\n제안과 채택\n\n\n7.2\n내용 변경\n\n\n7.2.1\n실질적인 변경\n\n\n7.3\n격하\n\n\n8\n같이 보기\n\n\n9\n외부 링크\n\n\n목차 토글\n\n위키백과:정책과 지침\n\n115개 언어"

In [13]:
# Chroma 벡터 저장소에 문서 저장하기
from langchain_chroma import Chroma
vector_store = Chroma(embedding_function=embedding_model)

# Document를 VectorStore에 저장
document_ids = vector_store.add_documents(splitted_docs)

# 결과 확인
print(f"저장된 Document 개수: {len(document_ids)}")

저장된 Document 개수: 5


In [14]:
print(f"저장된 Document ID: {document_ids[:5]}...")

저장된 Document ID: ['c8704b7d-1c12-4590-a385-64097413bf26', '2e34f3bb-9fd6-4457-8e0d-7326a71e030e', 'bed811a7-92a2-45df-90b1-620c077ef131', '2edb22e4-465d-411b-82fd-d9af20b9f302', 'c6479c8f-3190-4c02-bf39-08e1c212f43e']...


In [15]:
vector_store._collection.count()

5

In [16]:
# VectorStore에 저장된 Document 개수 확인
print(f"VectorStore에 저장된 Document 개수: {vector_store._collection.count()}")

VectorStore에 저장된 Document 개수: 5


> **참고**: Chroma 실행 시 `Failed to send telemetry event` 경고가 나타날 수 있습니다. 

- 이는 Chroma의 사용 통계 수집 기능과 관련된 것으로, 실제 기능에는 영향을 주지 않습니다. 무시하셔도 됩니다.

### Step 2: **Retrieval and generation**

`5. 검색 및 생성`

- `RunnableParallel`과 `RunnablePassthrough`를 조합하여 체인을 구성


In [17]:
# 벡터 스토어 문서 검색 - 유사도 기반 검색

search_query = "위키피디아 정책 변경 절차를 알려주세요"

results = vector_store.similarity_search(query=search_query, k=2)
for doc in results:
    print(f"* {doc.page_content} [{doc.metadata}]")
    print("-"*50)

* 링크 편집

프로젝트 문서토론

한국어

읽기원본 보기역사 보기

도구

도구
사이드바로 이동
숨기기

		작업
	


읽기

원본 보기

역사 보기


		일반
	


여기를 가리키는 문서가리키는 글의 최근 바뀜파일 올리기고유 링크문서 정보축약된 URL 얻기기존 파서로 전환

		인쇄/내보내기
	


책 만들기PDF로 다운로드인쇄용 판

		다른 프로젝트
	


추상 위키백과위키미디어 공용위키미디어 재단미디어위키메타위키위키미디어 아웃리치위키생물종위키책위키데이터위키인용집위키문헌위키배움터위키데이터 항목


외관
사이드바로 이동
숨기기


위키백과, 우리 모두의 백과사전.


이 문서는 한국어 위키백과의 정책입니다.이것은 모든 사용자들이 일반적으로 따라야 하는 널리 인정된 기준입니다. 문서의 변경은 총의를 반영해야 합니다.단축백:정책백:지침백:규칙백:방침WP:POLICY
요약: 위키백과의 정책과 지침은 위키백과 공동체가 지켜야 할 규범과 사례를 모아둔 것을 의미합니다. 본 정책은 정책과 지침이 어떻게 만들어지고 유지하며 지켜야 하는 지에 대해 설명합니다.
정책과 지침(목록)
원칙
다섯 원칙
규칙에 얽매이지 마세요

콘텐츠 정책(핵심)
중립적 시각
확인 가능
독자 연구 금지
위키백과에 대한 오해

행동 정책
총의
분쟁 해결
편집 분쟁
삭제 정책
차단 정책
법적 위협 금지
인신 공격 금지
문서의 소유권
계정 이름
문서 훼손

기타 정책 분류
집행 정책
법적 정책
절차 정책
생존 인물의 전기 [{'source': 'https://ko.wikipedia.org/wiki/%EC%9C%84%ED%82%A4%EB%B0%B1%EA%B3%BC:%EC%A0%95%EC%B1%85%EA%B3%BC_%EC%A7%80%EC%B9%A8', 'language': 'ko', 'title': '위키백과:정책과 지침 - 위키백과, 우리 모두의 백과사전'}]
--------------------------------------------------
* 위키백과:정책과 지침 - 위키백과, 우리 모

In [18]:
# 벡터 스토어 검색기 설정 - 유사도 기반 검색

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2},
)

# 검색기로 검색하기
results = retriever.invoke(input=search_query)

# 결과 확인
for doc in results:
    print(f"* {doc.page_content} [{doc.metadata}]")
    print("-"*50)

* 링크 편집

프로젝트 문서토론

한국어

읽기원본 보기역사 보기

도구

도구
사이드바로 이동
숨기기

		작업
	


읽기

원본 보기

역사 보기


		일반
	


여기를 가리키는 문서가리키는 글의 최근 바뀜파일 올리기고유 링크문서 정보축약된 URL 얻기기존 파서로 전환

		인쇄/내보내기
	


책 만들기PDF로 다운로드인쇄용 판

		다른 프로젝트
	


추상 위키백과위키미디어 공용위키미디어 재단미디어위키메타위키위키미디어 아웃리치위키생물종위키책위키데이터위키인용집위키문헌위키배움터위키데이터 항목


외관
사이드바로 이동
숨기기


위키백과, 우리 모두의 백과사전.


이 문서는 한국어 위키백과의 정책입니다.이것은 모든 사용자들이 일반적으로 따라야 하는 널리 인정된 기준입니다. 문서의 변경은 총의를 반영해야 합니다.단축백:정책백:지침백:규칙백:방침WP:POLICY
요약: 위키백과의 정책과 지침은 위키백과 공동체가 지켜야 할 규범과 사례를 모아둔 것을 의미합니다. 본 정책은 정책과 지침이 어떻게 만들어지고 유지하며 지켜야 하는 지에 대해 설명합니다.
정책과 지침(목록)
원칙
다섯 원칙
규칙에 얽매이지 마세요

콘텐츠 정책(핵심)
중립적 시각
확인 가능
독자 연구 금지
위키백과에 대한 오해

행동 정책
총의
분쟁 해결
편집 분쟁
삭제 정책
차단 정책
법적 위협 금지
인신 공격 금지
문서의 소유권
계정 이름
문서 훼손

기타 정책 분류
집행 정책
법적 정책
절차 정책
생존 인물의 전기 [{'source': 'https://ko.wikipedia.org/wiki/%EC%9C%84%ED%82%A4%EB%B0%B1%EA%B3%BC:%EC%A0%95%EC%B1%85%EA%B3%BC_%EC%A7%80%EC%B9%A8', 'language': 'ko', 'title': '위키백과:정책과 지침 - 위키백과, 우리 모두의 백과사전'}]
--------------------------------------------------
* 위키백과:정책과 지침 - 위키백과, 우리 모

In [19]:
# RunnableParallel을 사용한 RAG 체인

from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 시스템 프롬프트
system_prompt = (
    "다음 검색된 맥락을 사용하여 사용자의 질문에 답하세요. "
    "답을 모르면 모른다고 하고, 추측하지 마세요. "
    "답변은 한국어로 간결하고 정확하게 작성하세요.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

# LLM 모델 설정
llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)

rag_chain = RunnableParallel({
    "context": retriever | RunnableLambda(lambda docs: "\n\n".join([doc.page_content for doc in docs])),
    "input": RunnablePassthrough()
    }) | prompt | llm | StrOutputParser()

query = "위키피디아 정책 변경 절차를 알려주세요"
response = rag_chain.invoke(query)

# 결과 확인
print(f"\n답변:\n{response}")


답변:
위키백과 정책 변경 절차는 다음과 같습니다.

1. 제안: 정책 또는 지침 변경을 위해 제안서를 작성하거나 토론을 시작합니다.
2. 토론: 위키백과 사용자들이 토론 페이지에서 의견을 나누며 변경 내용에 대해 논의합니다.
3. 합의 형성: 충분한 논의와 의견 수렴 후, 대다수의 사용자들이 동의하는 방향으로 합의가 형성됩니다.
4. 채택: 합의된 변경 내용이 공식 정책 또는 지침으로 채택됩니다.
5. 문서 수정: 채택된 변경 사항을 정책 문서에 반영하여 업데이트합니다.

이 과정은 커뮤니티의 총의와 협의를 통해 이루어지며, 공식적인 절차와 규범에 따라 진행됩니다.


# [실습 프로젝트]

### RAG 파이프라인 구축하기

이제 배운 내용을 바탕으로 실제 RAG 파이프라인을 직접 구축해보겠습니다.

**목표**: 뉴스 기사를 로드하고, 청크로 분할하여 벡터 저장소에 저장한 후, 질문에 답변하는 RAG 시스템 구축

**단계별 가이드**:
1. 데이터 준비 → 텍스트 처리 → 임베딩 → 검색 → 평가
2. 각 과제마다 구현해야 할 함수와 결과물 정의
3. 단위 테스트를 통한 검증

In [20]:
# 1단계: 데이터 준비 - 웹문서 검색을 위해 관련 URL 가져오기
web_urls = [
    "https://n.news.naver.com/mnews/article/029/0002927209",
    "https://n.news.naver.com/mnews/article/092/0002358620",
    "https://n.news.naver.com/mnews/article/008/0005136824",
]

In [22]:
# 2단계: WebBaseLoader를 사용해 텍스트 로드
"""
힌트:
- WebBaseLoader를 사용하여 web_urls의 문서들을 로드하세요
- loader.load()를 호출하여 Document 객체 리스트를 얻습니다
- 로드된 문서의 개수를 출력하여 확인하세요

기대 출력:
- 로드된 문서 개수: 3개 (URL 개수와 동일)
"""
# Data Loader - 웹페이지 데이터 가져오기
from langchain_community.document_loaders import WebBaseLoader  # type: ignore

loader = WebBaseLoader(web_urls)

# 웹페이지 텍스트 -> Document 객체로 변환
document1 = loader.load() 

# 결과 확인
print(f"Document 개수: {len(document1)}")
print(f"Document 길이: {len(document1[0].page_content)}")
print(f"Document 내용: {document1[0].page_content[5000:6000]}")

Document 개수: 3
Document 길이: 10019
Document 내용: 인공지능, 로봇의 몸을 입다…피지컬 AI의 현재











#
증시


6개 언론사
3,440개 기사










롤러코스터 코스피



구독


구독중




오락가락 코스피 다시 ‘8000선’ 밑으로…4거래일 연속 사이드카













진격의 코스피



구독


구독중




“오락가락 코스피, 투자 못하겠네”…이달 거래량 올해들어 최저 [이런국장 저런주식]













오늘의 증시



구독


구독중




外人·기관 5조 던졌다…코스피 4.5% 하락 마감











#
윤석열


25개 언론사
44,049개 기사










2차 종합특검



구독


구독중




[단독] "윤, 미친 줄" 오른팔 김태효 '혼자 살려고' 원색적 비난


영상













2차 종합특검



구독


구독중




종합특검, '해경 내란가담 의혹' 김종욱 前해경청장 피의자 소환













윤석열·김건희 재판



구독


구독중




종합특검, ‘해경 계엄 가담 의혹’ 김종욱 전 해경청장 소환











#
중동전쟁


9개 언론사
5,811개 기사










美-이란 전쟁



구독


구독중




튀르키예서 오만까지…호르무즈 대신할 ‘헤자즈 철도’ 부활 추진













중동 전쟁



구독


구독중




[글로벌+] 종전 선언 신뢰감 잃어가는 미국···전쟁 영향 받는 북중미 월드컵


영상













美·이란 전쟁



구독


구독중




美·이란, 4월 휴전 후 최대 교전…IRGC "걸프 美거점 21곳 공격"











#
쿠팡


13개 언론사
3,884개 기사










쿠팡 개인정보 유출



구독


구독중




'3367만 건 개인정보 유출' 쿠팡 과징금, SKT 1348억 기록 깨나













쿠팡 사태 어디까지


In [23]:
# 3단계: CharacterTextSplitter로 문서 분할
"""
힌트:
- CharacterTextSplitter를 import하세요
- chunk_size=500, chunk_overlap=100으로 설정하세요
- separator="\n\n"로 문단 단위 분할
- split_documents() 메서드를 사용하여 문서를 분할하세요
- 분할된 청크의 개수를 출력하세요

기대 출력:
- 분할된 청크 개수: 약 20-40개 (문서 내용에 따라 다를 수 있음)
"""

# Text Split (Documents -> small chunks: Documents)
from langchain_text_splitters import CharacterTextSplitter  # type: ignore

# 1000자씩 잘라서 200자씩 겹치는 Document로 변환
text_splitter = CharacterTextSplitter(
    separator="\n\n",    # 문단 구분자
    chunk_size=500,     # 문단 길이
    chunk_overlap=100,   # 겹치는 길이
    length_function=len, # 길이 측정 함수
    is_separator_regex=False,   # separator가 정규식인지 여부
)

splitted_document1 = text_splitter.split_documents(document1)

# 결과 확인
print(f"Document 개수: {len(splitted_document1)}")
print("\n\n")

for i, doc in enumerate(splitted_document1):
    print(f"Document {i} 길이: {len(doc.page_content)}")
    print(f"Document {i} 내용: {doc.page_content[:100]}...")
    print("-"*50)

Created a chunk of size 1014, which is longer than the specified 500
Created a chunk of size 616, which is longer than the specified 500
Created a chunk of size 1526, which is longer than the specified 500
Created a chunk of size 703, which is longer than the specified 500


Document 개수: 62



Document 0 길이: 495
Document 0 내용: "그래서 AGI 왔다고?"…오픈AI 올트먼, X에 오묘한 메시지


본문 바로가기


NAVER

뉴스


엔터


스포츠


날씨


프리미엄


지방선거


검색


언론사별
...
--------------------------------------------------
Document 1 길이: 326
Document 1 내용: 입력
2025.01.05. 오후 5:48


수정
2025.01.05. 오후 5:50

기사원문
 


추천
반응


쏠쏠정보
0


흥미진진
0


공감백배
0


분석탁월
0
...
--------------------------------------------------
Document 2 길이: 1007
Document 2 내용: 샘 올트먼 'X' 갈무리.    오픈AI 설립자 샘 올트먼이 새해에 처음으로 인공지능(AI)의 미래에 대해 짧은 단상을 전했다. 올트먼은 4일 오후 1시(현지시간) X에 "나는 항...
--------------------------------------------------
Document 3 길이: 355
Document 3 내용: 김나인 기자(silkni@dt.co.kr)


Copyright ⓒ 디지털타임스. All rights reserved. 무단 전재 및 재배포 금지.

 
이 기사는 언론사에서 IT...
--------------------------------------------------
Document 4 길이: 476
Document 4 내용: 언론사홈


디지털타임스

			주요뉴스해당 언론사에서 선정하며 언론사(아웃링크)로 이동합니다.


영하 '강추위' 뚫고 알몸 마라톤 뛴 시민들…밝은 표정으로 완주
외로움이 사람 ...
--------------------------------------------------
Document 5 길이: 500
Document 5 내용: 네

In [ ]:
# 4단계: 임베딩 및 벡터 저장소 구현
"""
힌트:
- OpenAIEmbeddings를 사용하여 임베딩 모델을 생성하세요
- Chroma 벡터 저장소를 생성하세요
- add_documents()를 사용하여 분할된 문서를 벡터 저장소에 추가하세요
- 저장된 문서 개수를 확인하세요

기대 출력:
- 임베딩 모델 생성 완료
- 벡터 저장소에 저장된 문서 개수: (3단계의 청크 개수와 동일)
"""

# OpenAI Embeddings - 문장 임베딩
from langchain_openai import OpenAIEmbeddings  # type: ignore

# embedding model 생성
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",  # 사용할 모델 이름을 지정 가능 
)


임베딩 벡터의 차원: 1536
임베딩 벡터: [-0.00888824462890625, 0.04730224609375, 0.0105438232421875, 0.00147247314453125, -0.03497314453125, 0.04229736328125, -0.0011081695556640625, 0.0219573974609375, 0.032928466796875, -0.022308349609375]...


In [31]:
# Chroma 벡터 저장소에 문서 저장하기
from langchain_chroma import Chroma
naver_news_vector_store = Chroma(embedding_function=embedding_model)

# Document를 VectorStore에 저장
document_ids = naver_news_vector_store.add_documents(splitted_document1)

# 결과 확인
print(f"저장된 Document 개수: {len(document_ids)}")

저장된 Document 개수: 62


In [33]:
document_ids[0]

'71c0dfcc-a26d-4fce-be4b-dec8bb5b1629'

In [35]:
# 5단계: RAG 기반 QA 체인 구현
"""
힌트:
- ChatOpenAI로 LLM 모델을 생성하세요 (model="gpt-4.1-nano")
- ChatPromptTemplate으로 프롬프트 템플릿을 만드세요
- 벡터 저장소에서 as_retriever()로 검색기를 생성하세요
- RunnableParallel 사용하여 RAG 체인을 구성하세요

기대 출력:
- RAG 체인 생성 완료
"""

# RunnableParallel을 사용한 RAG 체인
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 시스템 프롬프트
system_prompt = (
    "다음 검색된 맥락을 사용하여 사용자의 질문에 답하세요. "
    "답을 모르면 모른다고 하고, 추측하지 마세요. "
    "답변은 한국어로 간결하고 정확하게 작성하세요.\n\n"
    "<맥락>{context}</맥락>"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "<질의>{input}</질의>")
])

# LLM 모델 설정
llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)



naver_news_retriever = naver_news_vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2},
)

# 검색기로 검색하기
naver_news_search_query = "AI 활용 방안에 대해서 알려주세요."
results = naver_news_retriever.invoke(input=naver_news_search_query)

# 결과 확인
for document1 in results:
    print("테스트")
    print(f"* {document1.page_content} [{document1.metadata}]")
    print("-"*50)


rag_chain = RunnableParallel({
    "context": naver_news_retriever | RunnableLambda(lambda docs: "\n\n".join([doc.page_content for doc in docs])),
    "input": RunnablePassthrough()
    }) | prompt | llm | StrOutputParser()



테스트
* "그래서 AGI 왔다고?"…오픈AI 올트먼, X에 오묘한 메시지


본문 바로가기


NAVER

뉴스


엔터


스포츠


날씨


프리미엄


지방선거


검색


언론사별


정치


경제


사회


생활/문화


IT/과학


세계


랭킹


신문보기


오피니언


TV


팩트체크


알고리즘 안내


정정보도 모음


MY


뉴스 이용 설정을 할 수 있어요


디지털타임스


구독

디지털타임스 언론사 구독되었습니다. 메인 뉴스판에서  주요뉴스를  볼 수 있습니다.
보러가기
닫기


디지털타임스 언론사 구독 해지되었습니다.
닫기


PICK
안내


언론사가 주요기사로선정한 기사입니다.
언론사별 바로가기
닫기


"그래서 AGI 왔다고?"…오픈AI 올트먼, X에 오묘한 메시지


입력
2025.01.05. 오후 5:48


수정
2025.01.05. 오후 5:50

기사원문
 


추천
반응


쏠쏠정보
0


흥미진진
0


공감백배
0


분석탁월
0 [{'title': '"그래서 AGI 왔다고?"…오픈AI 올트먼, X에 오묘한 메시지', 'source': 'https://n.news.naver.com/mnews/article/029/0002927209', 'language': 'ko'}]
--------------------------------------------------
테스트
* "그래서 AGI 왔다고?"…오픈AI 올트먼, X에 오묘한 메시지


본문 바로가기


NAVER

뉴스


엔터


스포츠


날씨


프리미엄


지방선거


검색


언론사별


정치


경제


사회


생활/문화


IT/과학


세계


랭킹


신문보기


오피니언


TV


팩트체크


알고리즘 안내


정정보도 모음


MY


뉴스 이용 설정을 할 수 있어요


디지털타임스


구독

디지털타임스 언론사 구독되었습니다. 메인 뉴스판에서  주요뉴스를  볼 수 있습니다.
보러가기
닫기


디지털타임스 언론사 구독 해지되었습니다

In [27]:
# 6단계: QA 체인으로 질문 응답
"""
힌트:
- 뉴스 기사와 관련된 질문을 준비하세요
- rag_chain.invoke({"question": 질문})으로 답변을 얻으세요
- response['answer']로 답변 내용을 확인하세요
- response['context']로 검색된 문서를 확인하세요

기대 출력:
- 질문에 대한 답변이 출력됩니다
- 검색된 관련 문서들의 내용이 포함됩니다

예시 질문:
- "기사의 주요 내용을 요약해주세요"
- "주요 인물은 누구인가요?"
- "어떤 사건에 대한 기사인가요?"
"""

query = "기사의 주요 내용을 요약해주세요."
response = rag_chain.invoke(query)

# 결과 확인
print(f"\n답변:\n{response}")



답변:
이 맥락에서는 2차 종합특검과 관련된 내용이 주로 다루어지고 있으며, 윤석열 정부와 관련된 수사, 해경 내란 가담 의혹, 김종욱 전 해경청장 소환, 그리고 미국과 이란 간의 군사적 긴장 상황 등이 언급되고 있습니다. 따라서, 주요 내용은 검찰 수사와 국제 군사 긴장 상황에 관한 것으로 보입니다.


In [36]:
response['answer']


TypeError: string indices must be integers, not 'str'